In [1]:
%%time
!pip uninstall -y torch
!pip install -q --no-index --find-links=/kaggle/input/making-wheels-of-necessary-packages-for-vllm vllm
!pip install -q -U /kaggle/input/vllm-t4-fix/grpcio-1.62.2-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
!pip install -q -U /kaggle/input/vllm-t4-fix/ray-2.11.0-cp310-cp310-manylinux2014_x86_64.whl
!pip install -q --no-deps --no-index /kaggle/input/hf-libraries/sentence-transformers/sentence_transformers-3.1.0-py3-none-any.whl
!pip install --no-deps --no-index /kaggle/input/logits-processor-zoo/logits_processor_zoo-0.1.0-py3-none-any.whl

Found existing installation: torch 2.4.0
Uninstalling torch-2.4.0:
  Successfully uninstalled torch-2.4.0
Processing /kaggle/input/logits-processor-zoo/logits_processor_zoo-0.1.0-py3-none-any.whl
CPU times: user 2.17 s, sys: 565 ms, total: 2.73 s
Wall time: 3min 35s


In [2]:
!pip install transformers peft accelerate \
    -q -U --no-index --find-links /kaggle/input/lmsys-wheel-files

In [3]:
%%capture
!pip install --no-index /kaggle/input/bitsandbytes0-42-0/bitsandbytes-0.42.0-py3-none-any.whl --find-links=/kaggle/input/bitsandbytes0-42-0
!pip install --no-index  /kaggle/input/bitsandbytes0-42-0/optimum-1.21.2-py3-none-any.whl --find-links=/kaggle/input/bitsandbytes0-42-0
!pip install --no-index  /kaggle/input/bitsandbytes0-42-0/auto_gptq-0.7.1-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl --find-links=/kaggle/input/bitsandbytes0-42-0

# 1.Recall use bge-large

In [4]:
TOPK=50

In [5]:
import os, math, numpy as np
import pandas as pd
import os
from transformers import AutoTokenizer
from tqdm import tqdm
import re, gc
import torch
from sentence_transformers import SentenceTransformer, util

os.environ["CUDA_VISIBLE_DEVICES"]="0,1"

In [6]:

IS_SUBMISSION = True

df_train = pd.read_csv("/kaggle/input/eedi-mining-misconceptions-in-mathematics/train.csv").fillna(-1).reset_index(drop=True)
df_test = pd.read_csv("/kaggle/input/eedi-mining-misconceptions-in-mathematics/test.csv")
df_misconception_mapping = pd.read_csv("/kaggle/input/eedi-mining-misconceptions-in-mathematics/misconception_mapping.csv")

if not IS_SUBMISSION:
    df_ret = df_train.sample(50, random_state=42).copy()
else:
    df_ret = df_test.copy()

In [7]:
def preprocess_text(x):
    x = x.lower()                 # Convert words to lowercase
    x = re.sub("@\w+", '',x)      # Delete strings starting with @
    #x = re.sub("'\d+", '',x)      # Delete Numbers
    x = re.sub("http\w+", '',x)   # Delete URL
    x = re.sub(r"\\\(", " ", x)
    x = re.sub(r"\\\)", " ", x)
    x = re.sub(r"[ ]{1,}", " ", x)
    x = re.sub(r"\.+", ".", x)    # Replace consecutive commas and periods with one comma and period character
    x = re.sub(r"\,+", ",", x)
    x = x.strip()                 # Remove empty characters at the beginning and end
    return x

In [8]:
rows = []

if not IS_SUBMISSION:
    for idx, row in df_ret.iterrows():
        for option in ["A", "B", "C", "D"]:
            if (row.CorrectAnswer==option) | (row[f"Misconception{option}Id"]==-1):
                continue
                
            correct_answer = row[f"Answer{row.CorrectAnswer}Text"]
            query_text =f"{row['SubjectName']}. {row['ConstructName']}. {row['QuestionText']}"
            rows.append({"query_text": query_text, 
                         "QuestionId_Answer": f"{row.QuestionId}_{option}",
                         "ConstructName": row.ConstructName,
                         "SubjectName": row.SubjectName,
                         "QuestionText": row.QuestionText,
                         "correct_answer": correct_answer,
                         "incorrect_answer": row[f"Answer{option}Text"]
                         })
else:
    for idx, row in df_ret.iterrows():
        for option in ["A", "B", "C", "D"]:
            if option == row.CorrectAnswer:
                continue
                
            correct_answer = row[f"Answer{row.CorrectAnswer}Text"]
    
            query_text =f"{row['SubjectName']}. {row['ConstructName']}. {row['QuestionText']}"
            rows.append({"query_text": query_text, 
                         "QuestionId_Answer": f"{row.QuestionId}_{option}",
                         "ConstructName": row.ConstructName,
                         "SubjectName": row.SubjectName,
                         "QuestionText": row.QuestionText,
                         "correct_answer": correct_answer,
                         "incorrect_answer": row[f"Answer{option}Text"]
                         })

df = pd.DataFrame(rows)
df['query_text'] = df['query_text'].apply(lambda x: preprocess_text(x))
df

,query_text,QuestionId_Answer,ConstructName,SubjectName,QuestionText,correct_answer,incorrect_answer
0,bidmas. use the order of operations to carry o...,1869_B,Use the order of operations to carry out calcu...,BIDMAS,\[\n3 \times 2+4-5\n\]\nWhere do the brackets ...,\( 3 \times(2+4)-5 \),\( 3 \times 2+(4-5) \)
1,bidmas. use the order of operations to carry o...,1869_C,Use the order of operations to carry out calcu...,BIDMAS,\[\n3 \times 2+4-5\n\]\nWhere do the brackets ...,\( 3 \times(2+4)-5 \),\( 3 \times(2+4-5) \)
2,bidmas. use the order of operations to carry o...,1869_D,Use the order of operations to carry out calcu...,BIDMAS,\[\n3 \times 2+4-5\n\]\nWhere do the brackets ...,\( 3 \times(2+4)-5 \),Does not need brackets
3,simplifying algebraic fractions. simplify an a...,1870_A,Simplify an algebraic fraction by factorising ...,Simplifying Algebraic Fractions,"Simplify the following, if possible: \( \frac{...",Does not simplify,\( m+1 \)
4,simplifying algebraic fractions. simplify an a...,1870_B,Simplify an algebraic fraction by factorising ...,Simplifying Algebraic Fractions,"Simplify the following, if possible: \( \frac{...",Does not simplify,\( m+2 \)
5,simplifying algebraic fractions. simplify an a...,1870_C,Simplify an algebraic fraction by factorising ...,Simplifying Algebraic Fractions,"Simplify the following, if possible: \( \frac{...",Does not simplify,\( m-1 \)
6,range and interquartile range from a list of d...,1871_A,Calculate the range from a list of data,Range and Interquartile Range from a List of Data,Tom and Katie are discussing the \( 5 \) plant...,Only\nKatie,Only\nTom
7,range and interquartile range from a list of d...,1871_C,Calculate the range from a list of data,Range and Interquartile Range from a List of Data,Tom and Katie are discussing the \( 5 \) plant...,Only\nKatie,Both Tom and Katie
8,range and interquartile range from a list of d...,1871_D,Calculate the range from a list of data,Range and Interquartile Range from a List of Data,Tom and Katie are discussing the \( 5 \) plant...,Only\nKatie,Neither is correct


In [9]:
recall_model_path = '/kaggle/input/baai/transformers/bge-large-en-v1.5/1' #0.52

model = SentenceTransformer(recall_model_path)
embedding_query = model.encode(df['query_text'], convert_to_tensor=True)
misconceptions_names = df_misconception_mapping.MisconceptionName.values
embedding_Misconception = model.encode(misconceptions_names, convert_to_tensor=True)

# the first time retrieval for LLM prompt
Ret_topNids = util.semantic_search(embedding_query, embedding_Misconception, top_k=TOPK)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/81 [00:00<?, ?it/s]

In [10]:
indices = []
for top_ids in Ret_topNids:
    indice = []
    for top_id in top_ids:
        indice.append(top_id['corpus_id'])
    indices.append(indice)
    
indices = np.array(indices)
indices.shape


(9, 50)

In [11]:
import gc

del model,embedding_query,embedding_Misconception

gc.collect()
torch.cuda.empty_cache()

In [12]:
np.save("indices_50_bge.npy", indices)
# df.to_parquet("df.parquet", index=False)

# recall use qwen14b

In [13]:
import pandas as pd
import os

full_df = pd.read_csv("/kaggle/input/eedi-mining-misconceptions-in-mathematics/test.csv")
IS_SUBMISSION = bool(os.getenv("KAGGLE_IS_COMPETITION_RERUN"))

rows = []
for idx, row in full_df.iterrows():
    for option in ["A", "B", "C", "D"]:
        if option == row.CorrectAnswer:
            continue
            
        correct_answer = row[f"Answer{row.CorrectAnswer}Text"]

        query_text =f"### SubjectName: {row['SubjectName']}\n### ConstructName: {row['ConstructName']}\n### Question: {row['QuestionText']}\n### Correct Answer: {correct_answer}\n### Misconcepte Incorrect answer: {option}.{row[f'Answer{option}Text']}"
        rows.append({"query_text": query_text, 
                     "QuestionId_Answer": f"{row.QuestionId}_{option}",
                     "ConstructName": row.ConstructName,
                     "SubjectName": row.SubjectName,
                     "QuestionText": row.QuestionText,
                     "correct_answer": correct_answer,
                     "incorrect_answer": row[f"Answer{option}Text"]
                     })

df = pd.DataFrame(rows)
df

,query_text,QuestionId_Answer,ConstructName,SubjectName,QuestionText,correct_answer,incorrect_answer
0,### SubjectName: BIDMAS\n### ConstructName: Us...,1869_B,Use the order of operations to carry out calcu...,BIDMAS,\[\n3 \times 2+4-5\n\]\nWhere do the brackets ...,\( 3 \times(2+4)-5 \),\( 3 \times 2+(4-5) \)
1,### SubjectName: BIDMAS\n### ConstructName: Us...,1869_C,Use the order of operations to carry out calcu...,BIDMAS,\[\n3 \times 2+4-5\n\]\nWhere do the brackets ...,\( 3 \times(2+4)-5 \),\( 3 \times(2+4-5) \)
2,### SubjectName: BIDMAS\n### ConstructName: Us...,1869_D,Use the order of operations to carry out calcu...,BIDMAS,\[\n3 \times 2+4-5\n\]\nWhere do the brackets ...,\( 3 \times(2+4)-5 \),Does not need brackets
3,### SubjectName: Simplifying Algebraic Fractio...,1870_A,Simplify an algebraic fraction by factorising ...,Simplifying Algebraic Fractions,"Simplify the following, if possible: \( \frac{...",Does not simplify,\( m+1 \)
4,### SubjectName: Simplifying Algebraic Fractio...,1870_B,Simplify an algebraic fraction by factorising ...,Simplifying Algebraic Fractions,"Simplify the following, if possible: \( \frac{...",Does not simplify,\( m+2 \)
5,### SubjectName: Simplifying Algebraic Fractio...,1870_C,Simplify an algebraic fraction by factorising ...,Simplifying Algebraic Fractions,"Simplify the following, if possible: \( \frac{...",Does not simplify,\( m-1 \)
6,### SubjectName: Range and Interquartile Range...,1871_A,Calculate the range from a list of data,Range and Interquartile Range from a List of Data,Tom and Katie are discussing the \( 5 \) plant...,Only\nKatie,Only\nTom
7,### SubjectName: Range and Interquartile Range...,1871_C,Calculate the range from a list of data,Range and Interquartile Range from a List of Data,Tom and Katie are discussing the \( 5 \) plant...,Only\nKatie,Both Tom and Katie
8,### SubjectName: Range and Interquartile Range...,1871_D,Calculate the range from a list of data,Range and Interquartile Range from a List of Data,Tom and Katie are discussing the \( 5 \) plant...,Only\nKatie,Neither is correct


In [14]:
import torch
from numpy.linalg import norm
import torch.nn.functional as F
from torch import Tensor
from transformers import AutoConfig, AutoTokenizer, AutoModelForMaskedLM, AutoModel, BitsAndBytesConfig
from peft import (
    LoraConfig,
    get_peft_model,
)
import sys
from tqdm import tqdm
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn.functional as F
import torch.nn as nn
import math
from sklearn.neighbors import NearestNeighbors

def batch_to_device(batch, target_device):
    """
    send a pytorch batch to a device (CPU/GPU)
    """
    for key in batch:
        if isinstance(batch[key], Tensor):
            batch[key] = batch[key].to(target_device)
    return batch

def last_token_pool(last_hidden_states: Tensor,
                    attention_mask: Tensor) -> Tensor:
    left_padding = (attention_mask[:, -1].sum() == attention_mask.shape[0])
    if left_padding:
        return last_hidden_states[:, -1]
    else:
        sequence_lengths = attention_mask.sum(dim=1) - 1
        batch_size = last_hidden_states.shape[0]
        return last_hidden_states[torch.arange(batch_size, device=last_hidden_states.device), sequence_lengths]

def get_detailed_instruct(task_description: str, query: str) -> str:
    return f'Instruct: {task_description}\nQuery: {query}'

def inference(df, model, tokenizer, device):
    batch_size = 16
    max_length = 512
    sentences = list(df['query_text'].values)

    all_embeddings = []
    length_sorted_idx = np.argsort([-len(sen) for sen in sentences])
    sentences_sorted = [sentences[idx] for idx in length_sorted_idx]
    for start_index in trange(0, len(sentences), batch_size, desc="Batches", disable=False):
        sentences_batch = sentences_sorted[start_index: start_index + batch_size]
        features = tokenizer(sentences_batch, max_length=max_length, padding=True, truncation=True,
                             return_tensors="pt")
        features = batch_to_device(features, device)
        with torch.no_grad():
            outputs = model(**features)
            embeddings = last_token_pool(outputs.last_hidden_state, features['attention_mask'])
            embeddings = torch.nn.functional.normalize(embeddings, dim=-1)
            embeddings = embeddings.detach().cpu().numpy().tolist()
        all_embeddings.extend(embeddings)

    all_embeddings = [np.array(all_embeddings[idx]).reshape(1, -1) for idx in np.argsort(length_sorted_idx)]

    return np.concatenate(all_embeddings, axis=0)

In [15]:
path_prefix = "/kaggle/input/eedi-mining-misconceptions-in-mathematics"
model_path = "/kaggle/input/qwen2.5-14/pytorch/default/1"

lora_path='/kaggle/input/qwen14b-it-lora/lora_weights/adapter.bin'
device='cuda:0'

In [16]:
tokenizer = AutoTokenizer.from_pretrained(lora_path.replace("/adapter.bin",""))
bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16
        )
model_base = AutoModel.from_pretrained(model_path, 
                                  quantization_config=bnb_config, 
                                  device_map=device,
                                  trust_remote_code=True)

if lora_path:
    print("loading lora")
    config = LoraConfig(
        r=64,
        lora_alpha=128,
        target_modules=[
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            "gate_proj",
            "up_proj",
            "down_proj",
        ],
        bias="none",
        lora_dropout=0.05,  # Conventional
        task_type="FEATURE_EXTRACTION",
    )
    model = get_peft_model(model_base, config)
    d = torch.load(lora_path, map_location=model.device)
    model.load_state_dict(d, strict=False)
    model = model.merge_and_unload()
model = model.eval()

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

loading lora


/opt/conda/lib/python3.10/site-packages/peft/tuners/lora/bnb.py:325: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


In [17]:
import numpy as np
from tqdm.autonotebook import trange


task_description = 'Given a math question with correct answer and a misconcepted incorrect answer, retrieve the most accurate misconception for the incorrect answer.'
df['query_text'] = df['query_text'].apply(lambda x: get_detailed_instruct(task_description,x))
V_answer = inference(df, model, tokenizer, device)

misconception_df = pd.read_csv("/kaggle/input/eedi-mining-misconceptions-in-mathematics/misconception_mapping.csv")
misconception_df["query_text"] = misconception_df["MisconceptionName"]
if not IS_SUBMISSION:
    misconception_df = misconception_df[:100]


V_misconception = inference(misconception_df, model, tokenizer, device)

V_answer.shape,V_misconception.shape

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

We detected that you are passing `past_key_values` as a tuple and this is deprecated and will be removed in v4.43. Please use an appropriate `Cache` class (https://huggingface.co/docs/transformers/v4.41.3/en/internal/generation_utils#transformers.Cache)


Batches:   0%|          | 0/7 [00:00<?, ?it/s]

((9, 5120), (100, 5120))

In [18]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def get_matches(V_topic, V_content, n_neighbors=25):
    
    neighbors_model = NearestNeighbors(n_neighbors=n_neighbors, metric='cosine', algorithm="brute", n_jobs=-1)
    neighbors_model.fit(V_content)
    dists, indices = neighbors_model.kneighbors(V_topic)
    
    return indices

indices = get_matches(V_answer, V_misconception, n_neighbors=50)
indices.shape

(9, 50)

In [19]:
import gc

del model_base, model, tokenizer, d

gc.collect()
torch.cuda.empty_cache()

In [20]:
np.save("indices_50_qwen14B.npy", indices)
df.to_parquet("df.parquet", index=False)

# 2.Picking the best candidate using qwen-32b-instruct-awq

25个答案，分3次迭代，每次迭代从8个答案加上次的survior（共9个)中，选出一个新的survior

In [21]:
%%writefile run_vllm.py

import vllm
import numpy as np
import pandas as pd
from transformers import PreTrainedTokenizer, AutoTokenizer
from typing import List
import torch
from logits_processor_zoo.vllm import MultipleChoiceLogitsProcessor
import re



def preprocess_text(x):
    x = re.sub("http\w+", '',x)   # Delete URL
    x = re.sub(r"\.+", ".", x)    # Replace consecutive commas and periods with one comma and period character
    x = re.sub(r"\,+", ",", x)
    x = re.sub(r"\\\(", " ", x)
    x = re.sub(r"\\\)", " ", x)
    x = re.sub(r"[ ]{1,}", " ", x)
    x = x.strip()                 # Remove empty characters at the beginning and end
    return x

# PROMPT  = """Here is a question about {ConstructName}({SubjectName}).
# Question: {Question}
# Correct Answer: {CorrectAnswer}
# Incorrect Answer: {IncorrectAnswer}

# You are a Mathematics teacher. Your task is to reason and identify the misconception behind the Incorrect Answer with the Question.
# Answer concisely what misconception it is to lead to getting the incorrect answer.
# Pick the correct misconception number from the below:

# {Retrival}
# """


PROMPT  = """A Diagnostic Question is a multiple-choice question with two options: one correct answer and one incorrect answer. The incorrect Answer is carefully crafted to capture a specific misconception. For example:
### Question: \( 1.39+2.53= \) 
### Correct Answer: \( 3.92 \) 
### Incorrect Answer: \( 4.01 \)
If a student selects the incorrect Answer: "\( 4.01 \)", they may have the misconception "When adding decimals, just adds the digits and ignores place value.".

Here is a dignostic question about measuring the student ability to {ConstructName}({SubjectName}).
### Question: {Question}
### Correct Answer: {CorrectAnswer}
### Incorrect Answer: {IncorrectAnswer}

You are a Mathematics teacher. Your task is to reason and identify the misconception behind the Incorrect Answer to the Question.
Compare the affinity between misconceptions and the incorrect answer. Answer concisely what misconception it is to lead to getting the incorrect answer.
Pick the misconception number from the below:

{Retrival}
"""


def apply_template(row, tokenizer):
    messages = [
        {
            "role": "user", 
            "content": preprocess_text(
                PROMPT.format(
                    ConstructName=row["ConstructName"],
                    SubjectName=row["SubjectName"],
                    Question=row["QuestionText"],
                    IncorrectAnswer=row[f"incorrect_answer"],
                    CorrectAnswer=row[f"correct_answer"],
                    Retrival=row[f"retrieval"]
                )
            )
        }
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return text


misconception_df = pd.read_csv("/kaggle/input/eedi-mining-misconceptions-in-mathematics/misconception_mapping.csv")

df = pd.read_parquet("df.parquet")
indices_50_qwen14B = np.load("indices_50_qwen14B.npy")
indices_50_bge = np.load("indices_50_bge.npy")

model_path = "/kaggle/input/qwen2.5/transformers/32b-instruct-awq/1"

llm = vllm.LLM(
    model_path,
    quantization="awq",
    tensor_parallel_size=2,
    gpu_memory_utilization=0.90, 
    trust_remote_code=True,
    dtype="half", 
    enforce_eager=True,
    max_model_len=5120,
    disable_log_stats=True
)
tokenizer = llm.get_tokenizer()


def get_candidates(c_indices):
    candidates = []

    mis_names = misconception_df["MisconceptionName"].values
    for ix in c_indices:
        c_names = []
        for i, name in enumerate(mis_names[ix]):
            c_names.append(f"{i+1}. {name}")

        candidates.append("\n".join(c_names))
        
    return candidates

survive_num = 3 #min=1
survivors_list = []
indices_50 = indices_50_qwen14B
indices = indices_50[:, :25]
survivors = indices[:, -1:]
for num in range(survive_num):
    if num == 0:
        indices = indices_50[:, :25]
        survivors = indices[:, -1:]
    else:
        indices_temp = np.zeros((indices_50.shape[0], 25),dtype=int)
        for i in range(indices_temp.shape[0]):
            ix = indices[i]
            llm_choice = survivors[i, 0]
            new_indice = indices_50[i, 25+num-1]
            indices_temp[i] = np.array([x for x in ix if x != llm_choice] + [new_indice])
        indices = indices_temp
        survivors = indices[:, -1:]
    for i in range(3):
        c_indices = np.concatenate([indices[:, -8*(i+1)-1:-8*i-1], survivors], axis=1)
        
        df["retrieval"] = get_candidates(c_indices)
        df["text"] = df.apply(lambda row: apply_template(row, tokenizer), axis=1)
        
        print("Example:")
        print(df["text"].values[0])
        print()
        
        responses = llm.generate(
            df["text"].values,
            vllm.SamplingParams(
                n=1,  # Number of output sequences to return for each prompt.
                top_k=1,  # Float that controls the cumulative probability of the top tokens to consider.
                temperature=0,  # randomness of the sampling
                seed=777, # Seed for reprodicibility
                skip_special_tokens=False,  # Whether to skip special tokens in the output.
                max_tokens=1,  # Maximum number of tokens to generate per output sequence.
                logits_processors=[MultipleChoiceLogitsProcessor(tokenizer, choices=["1", "2", "3", "4", "5", "6", "7", "8", "9"])]
            ),
            use_tqdm=True
        )
        responses = [x.outputs[0].text for x in responses]
        df["response"] = responses
        llm_choices = df["response"].astype(int).values - 1
        survivors = np.array([cix[best] for best, cix in zip(llm_choices, c_indices)]).reshape(-1, 1)
    
    survivors_list.append(survivors)



results = []
indices_50 = indices_50_bge
indices = indices_50[:, :25]
for i in range(indices.shape[0]):
    ix = indices[i]
    llm_choice_str_list = []
    llm_choice_list = []
    for num in range(survive_num):
        llm_choice_str_list.append(str(survivors_list[num][i,0]))
        llm_choice_list.append(survivors_list[num][i,0])

    res = llm_choice_str_list + [str(x) for x in ix if x not in llm_choice_list]
    res = res[:25]
    results.append(" ".join(res))


df["MisconceptionId"] = results
df.to_csv("submission.csv", columns=["QuestionId_Answer", "MisconceptionId"], index=False)

Writing run_vllm.py


In [22]:
!python run_vllm.py

/opt/conda/lib/python3.10/pty.py:89: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


WARNING 12-12 02:44:05 config.py:246] awq quantization is not fully optimized yet. The speed can be slower than non-quantized models.
INFO 12-12 02:44:05 config.py:715] Defaulting to use mp for distributed inference
INFO 12-12 02:44:05 llm_engine.py:176] Initializing an LLM engine (v0.5.3.post1) with config: model='/kaggle/input/qwen2.5/transformers/32b-instruct-awq/1', speculative_config=None, tokenizer='/kaggle/input/qwen2.5/transformers/32b-instruct-awq/1', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, rope_scaling=None, rope_theta=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.float16, max_seq_len=5120, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=2, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=awq, enforce_eager=True, kv_cache_dtype=auto, quantization_param_path=None, device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='outlines'), observability_config=ObservabilityConfig

In [23]:
pd.read_csv("submission.csv")

,QuestionId_Answer,MisconceptionId
0,1869_B,74 77 2306 987 2586 2518 466 1666 2221 2488 11...
1,1869_C,74 15 2306 987 2586 2518 466 1666 2221 2488 11...
2,1869_D,74 15 2306 987 2586 2518 466 1666 2221 2488 11...
3,1870_A,91 29 1540 2398 1593 979 2307 59 363 715 633 8...
4,1870_B,91 79 1540 2398 1593 979 2307 59 363 715 633 8...
5,1870_C,91 29 1540 2398 1593 979 2307 59 363 715 633 8...
6,1871_A,14 23 632 188 365 1059 1287 2319 2151 1073 397...
7,1871_C,23 74 632 188 365 1059 1287 2319 2151 1073 397...
8,1871_D,0 23 632 188 365 1059 1287 2319 2151 1073 397 ...
